# Perfilado de bronze: accidentes en carreteras

Análisis exploratorio para definir las reglas de limpieza de silver. No forma parte del pipeline y no escribe archivos.

- **Entrada:** `data/bronze/accidentes.parquet`
- **Resultado:** tabla de hallazgos al final

## Configuración

In [17]:
import pandas as pd
from pathlib import Path

origen = Path("../data/bronze/accidentes.parquet")

pd.set_option("display.max_columns", None)

In [18]:
df = pd.read_parquet(origen)
df.shape

(60531, 22)

## Estructura

In [19]:
df.head(3)

,id,Fecha,hora,uf,municipio,causa_accidente,tipo_acidente,clasificación_accidente,sentido_via,tipo_pista,Personas,Muertos,Herido_Leves,Heridos_graves,ilesos,ignorados,Heridos,vehículos,latitude,longitude,_fecha_carga,_archivo_origen
0,262002,2020-01-10,21:40:00,SP,QUELUZ,Falta de atención de conducción,Colisión trasera,Con víctimas heridas,Creciente,Dupla,2,0,1,0,1,0,1,2,-22.531582,-44.753747,2026-10-06 21:25:13.440021,BD-Accidentes-Carreteras.xlsx
1,264173,2020-01-21,08:05:00,SP,JACAREI,Falta de atención de conducción,Colisión trasera,Con víctimas heridas,DeCreciente,Dupla,2,0,1,0,1,0,1,2,-23.260721,-45.951792,2026-10-06 21:25:13.440021,BD-Accidentes-Carreteras.xlsx
2,270555,2020-02-20,21:40:00,SP,SANTA ISABEL,Falta de atención de conducción,Colisión trasera,Con víctimas heridas,DeCreciente,Dupla,2,0,1,0,1,0,1,2,-23.366930,-46.230812,2026-10-06 21:25:13.440021,BD-Accidentes-Carreteras.xlsx


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60531 entries, 0 to 60530
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id                       60531 non-null  int64         
 1   Fecha                    60531 non-null  datetime64[us]
 2   hora                     60531 non-null  object        
 3   uf                       60531 non-null  str           
 4   municipio                60531 non-null  str           
 5   causa_accidente          60531 non-null  str           
 6   tipo_acidente            60531 non-null  str           
 7   clasificación_accidente  60531 non-null  str           
 8   sentido_via              60531 non-null  str           
 9   tipo_pista               60531 non-null  str           
 10  Personas                 60531 non-null  int64         
 11  Muertos                  60531 non-null  int64         
 12  Herido_Leves             60531 non-null  in

In [21]:
# Periodo esperado: 2018 a 2020
df["Fecha"].min(), df["Fecha"].max()

(Timestamp('2018-01-01 00:00:00'), Timestamp('2020-12-31 00:00:00'))

In [22]:
# pandas muestra hora como object, pero los valores son de tipo time
df["hora"].head(3).tolist()

[datetime.time(21, 40), datetime.time(8, 5), datetime.time(21, 40)]

## Completitud y unicidad

In [23]:
# Nulos por columna
df.isna().sum()

id                         0
Fecha                      0
hora                       0
uf                         0
municipio                  0
causa_accidente            0
tipo_acidente              0
clasificación_accidente    0
sentido_via                0
tipo_pista                 0
Personas                   0
Muertos                    0
Herido_Leves               0
Heridos_graves             0
ilesos                     0
ignorados                  0
Heridos                    0
vehículos                  0
latitude                   0
longitude                  0
_fecha_carga               0
_archivo_origen            0
dtype: int64

In [24]:
# Filas duplicadas, sin contar las columnas de auditoría
df.drop(columns=["_fecha_carga", "_archivo_origen"]).duplicated().sum()

np.int64(0)

In [25]:
# Llave: id
df.duplicated(subset=["id"]).sum()

np.int64(0)

## Categorías

In [26]:
df.nunique()

id                         60531
Fecha                       1096
hora                        1297
uf                             4
municipio                    388
causa_accidente               46
tipo_acidente                 19
clasificación_accidente        3
sentido_via                    3
tipo_pista                     3
Personas                      58
Muertos                       11
Herido_Leves                  39
Heridos_graves                18
ilesos                        48
ignorados                      9
Heridos                       44
vehículos                     14
latitude                   33726
longitude                  33817
_fecha_carga                   1
_archivo_origen                1
dtype: int64

In [27]:
categorias = [
    "uf",
    "causa_accidente",
    "tipo_acidente",
    "clasificación_accidente",
    "sentido_via",
    "tipo_pista",
]

for col in categorias:
    print(df[col].value_counts(dropna=False).sort_index(), "\n")

uf
ES     8018
MG    26160
RJ    13417
SP    12936
Name: count, dtype: int64 

causa_accidente
Acceder a la vía sin observar la presencia de otros vehículos.                          2
Acceso irregular                                                                        1
Acumulación de aceite en el suelo.                                                      1
Acumulación de arena o escombros en el pavimento                                        1
Adelantamiento indebido                                                               783
Agresión externa                                                                      400
Animales en la pista                                                                 1019
Ausencia de señalización                                                                1
Carga excesiva y/o mal embalada                                                       386
Condutor Dormindo                                                                    2714
Consu

## Consistencia

Filas que incumplen cada regla. Esperado: 0.

In [28]:
# heridos = heridos leves + heridos graves
(df["Heridos"] != df["Herido_Leves"] + df["Heridos_graves"]).sum()

np.int64(0)

In [29]:
# personas = muertos + heridos + ilesos + ignorados
(df["Personas"] != df["Muertos"] + df["Heridos"] + df["ilesos"] + df["ignorados"]).sum()

np.int64(0)

In [30]:
# Rangos: sin negativos ni coordenadas fuera de Brasil
df.describe().loc[["min", "max"]]

,id,Fecha,Personas,Muertos,Herido_Leves,Heridos_graves,ilesos,ignorados,Heridos,vehículos,latitude,longitude,_fecha_carga
min,99973.0,2018-01-01 00:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-25.078084,-51.863960,2026-10-06 21:25:13.440021
max,364457.0,2020-12-31 00:00:00,80.0,13.0,58.0,20.0,62.0,11.0,66.0,16.0,-15.709300,-39.785442,2026-10-06 21:25:13.440021


## Hallazgos

Sin nulos, sin duplicados y sin espacios sobrantes. Las reglas de consistencia se cumplen en todas las filas. Corregir lo siguiente en Silver:

| Categoría | Columna | Problema | Acción en silver |
|---|---|---|---|
| Renombrar | Todas | Nombres con mayúsculas, acentos y restos de portugués (`uf`, `tipo_acidente`, `Herido_Leves`, `vehículos`, `latitude`, `longitude`) | Renombrar a minúsculas, sin acentos y en español |
| Renombrar | `tipo_pista` | Nombre de columna en portugués | Renombrar a `tipo_carril` |
| Tipo de dato | `Fecha` | En la fuente es solo fecha; pandas la carga como `datetime64` con hora 00:00 | Devolver a tipo fecha |
| Reemplazar texto | `causa_accidente` | Valores en portugués: "Condutor Dormindo", "Transitar no acostamento" | Traducir |
| Reemplazar texto | `causa_accidente` | Traducción literal errada: "Objeto estático en la plataforma del vagón", "No mantengas una distancia segura." | Corregir el texto |
| Reemplazar texto | `causa_accidente` | Algunos valores terminan en punto y otros no; "Fenomenos" sin tilde | Quitar el punto final y corregir ortografía |
| Reemplazar texto | `sentido_via` | "DeCreciente" con mayúscula intermedia | Corregir a "Decreciente" |
| Reemplazar texto | `tipo_acidente` | Valores en portugués: "Queda de ocupante de veículo" | Traducir |
| Reemplazar texto | `tipo_acidente` | Traducción literal errada: "Salida de la cama del carro" (8.774), "Listado" (6.004), "Peatón atropellando", "Posibles daños", "Estancamiento", "Desconexión de carga" | Corregir el texto según el catálogo de tipos de accidente de la PRF |
| Reemplazar texto | `tipo_pista` | "Dupla" en portugués | Traducir a "Doble" |
| Unificar | `causa_accidente` | Causas repetidas con otra redacción: "Consumo de alcohol" (3.193) y "Consumo de alcohol por parte del conductor" (3); "Enfermedad repentina" (730) y "Enfermedad repentina del conductor" (1) | Unificar. Confirmar con negocio |